In [1]:
import numpy as np
import time 
import optax
import sys 
import os 
import jax
import jax.numpy as jnp
from jax import value_and_grad, vmap, jit
from sklearn.metrics import mean_squared_error

from openmm.app import PDBFile
from openmm.unit import angstrom
from openmm.app import CutoffPeriodic
from functools import partial
import pickle

from dmff.api import Hamiltonian
from dmff.utils import jit_condition
from dmff.common import nblist

# from tools import *

from jax import config
# config.update("jax_enable_x64", True)
config.update("jax_debug_nans", True)  # Enable NaN checking

import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 200

os.environ['MPLCONFIGDIR'] = os.getcwd() + "/configs/"

In [2]:
class BasePairs:
    def __init__(self, ff, pdb, pdb_A, pdb_B):
        pdb = PDBFile(pdb)
        pdb_A = PDBFile(pdb_A)
        pdb_B = PDBFile(pdb_B)
        self.H = Hamiltonian(ff)
        self.pots = self.H.createPotential(pdb.topology, nonbondedCutoff=25*angstrom, nonbondedMethod=CutoffPeriodic, ethresh=1e-4)
        self.pots_A = self.H.createPotential(pdb_A.topology, nonbondedCutoff=25*angstrom, nonbondedMethod=CutoffPeriodic, ethresh=1e-4)
        self.pots_B = self.H.createPotential(pdb_B.topology, nonbondedCutoff=25*angstrom, nonbondedMethod=CutoffPeriodic, ethresh=1e-4)

        self.pos = jnp.array(pdb.positions._value)
        self.pos_A = jnp.array(pdb_A.positions._value) 
        self.pos_B = jnp.array(pdb_B.positions._value)

        self.box = jnp.eye(3) * 6
        self.rc = 2.5
        self.nblist = nblist.NeighborList(self.box, self.rc, self.pots.meta['cov_map'])
        self.nblist_A = nblist.NeighborList(self.box, self.rc, self.pots_A.meta['cov_map'])
        self.nblist_B = nblist.NeighborList(self.box, self.rc, self.pots_B.meta['cov_map'])
        self.nblist.allocate(self.pos)
        self.nblist_A.allocate(self.pos_A)
        self.nblist_B.allocate(self.pos_B)
        self.pairs = self.nblist.pairs
        self.pairs_A = self.nblist_A.pairs
        self.pairs_B = self.nblist_B.pairs
        self.pairs_AB = self.pairs[self.pairs[:, 0] < self.pairs[:, 1]]
        self.pairs_A = self.pairs_A[self.pairs_A[:, 0] < self.pairs_A[:, 1]]
        self.pairs_B = self.pairs_B[self.pairs_B[:, 0] < self.pairs_B[:, 1]]

        self.potentials_names = ['ex', 'sr_es', 'sr_pol', 'sr_disp', 'dhf', 'dmp_es', 'dmp_disp']
        self.potentials_mapping = {
            'ex': 'SlaterExForce',
            'sr_es': 'SlaterSrEsForce',
            'sr_pol': 'SlaterSrPolForce',
            'sr_disp': 'SlaterSrDispForce',
            'dhf': 'SlaterDhfForce',
            'dmp_es': 'QqTtDampingForce',
            'dmp_disp': 'SlaterDampingForce',
        }
        
        for potentials_name in self.potentials_names:
            setattr(self, f'pots_{potentials_name}', self.pots.dmff_potentials[self.potentials_mapping[potentials_name]])
            setattr(self, f'pots_{potentials_name}_A', self.pots_A.dmff_potentials[self.potentials_mapping[potentials_name]])
            setattr(self, f'pots_{potentials_name}_B', self.pots_B.dmff_potentials[self.potentials_mapping[potentials_name]])

    def cal_E(self, params0, pos_A, pos_B):
        params = params_convert(params0)
        # get position array
        pos_A *= 0.1
        pos_B *= 0.1
        pos_AB = jnp.concatenate([pos_A, pos_B], axis=0)
        box = self.box
        #####################
        # exchange repulsion
        #####################
        E_ex = self.pots_ex(pos_AB, box, self.pairs_AB, params)\
               - self.pots_ex_A(pos_A, box, self.pairs_A, params)\
               - self.pots_ex_B(pos_B, box, self.pairs_B, params)

        #######################
        # electrostatic
        #######################
        E_dmp_es = self.pots_dmp_es(pos_AB, box, self.pairs_AB, params) \
                    - self.pots_dmp_es_A(pos_A, box, self.pairs_A, params) \
                    - self.pots_dmp_es_B(pos_B, box, self.pairs_B, params)
        E_sr_es = self.pots_sr_es(pos_AB, box, self.pairs_AB, params) \
                - self.pots_sr_es_A(pos_A, box, self.pairs_A, params) \
                - self.pots_sr_es_B(pos_B, box, self.pairs_B, params)

        ###################################
        # polarization (induction) energy
        ###################################
        E_sr_pol = self.pots_sr_pol(pos_AB, box, self.pairs_AB, params) \
                    - self.pots_sr_pol_A(pos_A, box, self.pairs_A, params) \
                    - self.pots_sr_pol_B(pos_B, box, self.pairs_B, params)

        #############
        # dispersion
        #############
        E_dmp_disp = self.pots_dmp_disp(pos_AB, box, self.pairs_AB, params) \
                    - self.pots_dmp_disp_A(pos_A, box, self.pairs_A, params) \
                    - self.pots_dmp_disp_B(pos_B, box, self.pairs_B, params)
        E_sr_disp = self.pots_sr_disp(pos_AB, box, self.pairs_AB, params) \
                    - self.pots_sr_disp_A(pos_A, box, self.pairs_A, params) \
                    - self.pots_sr_disp_B(pos_B, box, self.pairs_B, params)

        ###########
        # dhf
        ###########
        E_dhf = self.pots_dhf(pos_AB, box, self.pairs_AB, params) \
                 - self.pots_dhf_A(pos_A, box, self.pairs_A, params) \
                 - self.pots_dhf_B(pos_B, box, self.pairs_B, params)

        E_es = E_dmp_es + E_sr_es
        E_pol = E_sr_pol
        E_disp = E_dmp_disp + E_sr_disp
        E_tot = E_ex + E_es + E_pol + E_disp + E_dhf
        return E_ex, E_es, E_pol, E_disp, E_dhf, E_tot 


In [3]:

# return data needed to fitting
def get_all_relavent_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a in arr or b in arr:
            dimer_test.append(key)
        elif b in arr or a in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

def get_all_contain_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a in arr and b in arr:
            dimer_test.append(key)
        elif b in arr and a in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

def get_all_homo_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a == b and b in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

# return data needed to fitting
def get_data_key(data, salt, solvent):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a == salt and b in solvent:
            dimer_test.append(key)
        elif b == salt and a in solvent:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

In [4]:
def calculate_rmsd(energy, energy_ref):
    energy = np.array(energy).T
    energy_ref = np.array(energy_ref).T
    rmsd_values = [np.sqrt(np.average((energy[i] - energy_ref[i])**2)) for i in range(len(energy))]
    return rmsd_values

def check_output_decompose(dimer_train, params):
    energies_pred = []
    energies_ref = []
    energy = []
    energy_ref = []
    shift = []
    # params = get_params(save_model, params0)
    for key in dimer_train:
        for sid in data[key].keys():
            scan_res = data[key][sid]
            tot_full = scan_res['tot_full']
            weights_pts = scan_res['wts']
            pos_A = jnp.array(scan_res['posA'])
            pos_B = jnp.array(scan_res['posB'])
            E_ex, E_es, E_pol, E_disp, E_dhf, E_tot = cal_energy[key](params, pos_A, pos_B)
            scan_res['sr_ex'], \
            scan_res['sr_es'], \
            scan_res['sr_pol'], \
            scan_res['sr_disp'], \
            scan_res['sr_dhf'],\
            scan_res['sr_ff'] = E_ex, E_es, E_pol, E_disp, E_dhf, E_tot
            E_ref = scan_res
            npts = len(E_tot)
            for ipt in range(npts):
                # if weights_pts[ipt] > 1e-2:
                # if tot_full[ipt] < 25:
                energy.append([E_tot[ipt],E_ex[ipt],E_es[ipt],E_pol[ipt],E_disp[ipt],E_dhf[ipt]])
                energy_ref.append([E_ref['tot'][ipt],E_ref['ex'][ipt],E_ref['es'][ipt],E_ref['pol'][ipt],E_ref['disp'][ipt],E_ref['dhf'][ipt]])
                shift.append(scan_res['shift'][ipt])
                if tot_full[ipt] < 25:
                    energies_pred.append([E_tot[ipt],E_ex[ipt],E_es[ipt],E_pol[ipt],E_disp[ipt],E_dhf[ipt]])
                    energies_ref.append([E_ref['tot'][ipt],E_ref['ex'][ipt],E_ref['es'][ipt],E_ref['pol'][ipt],E_ref['disp'][ipt],E_ref['dhf'][ipt]])
        rmsd_values = calculate_rmsd(energy, energy_ref)
        rmsd_midrange_value = calculate_rmsd(energies_pred, energies_ref)
        # print(key, '%.3f'%rmsd_values[0])
    return np.array(energy), np.array(energy_ref), np.array(rmsd_values), np.array(rmsd_midrange_value), np.array(shift)


def check_output_wb97(dimer_train, params):
    energies_pred = []
    energies_ref = []
    energy = []
    energy_ref = []
    shift = []
    # params = get_params(save_model, params0)
    for key in dimer_train:
        for sid in data[key].keys():
            scan_res = data[key][sid]
            tot_full = scan_res['tot_full']
            weights_pts = scan_res['wts']
            pos_A = jnp.array(scan_res['posA'])
            pos_B = jnp.array(scan_res['posB'])
            E_ex, E_es, E_pol, E_disp, E_dhf, E_tot = cal_energy[key](params, pos_A, pos_B)
            scan_res['sr_ex'], \
            scan_res['sr_es'], \
            scan_res['sr_pol'], \
            scan_res['sr_disp'], \
            scan_res['sr_dhf'],\
            scan_res['sr_ff'] = E_ex, E_es, E_pol, E_disp, E_dhf, E_tot
            E_ref = scan_res
            npts = len(E_tot)
            for ipt in range(npts):
                # if weights_pts[ipt] > 1e-2:
                # if tot_full[ipt] < 25:
                energy.append([E_tot[ipt]])
                energy_ref.append([E_ref['tot'][ipt]])
                shift.append(scan_res['shift'][ipt])
                if tot_full[ipt] < 25:
                    energies_pred.append([E_tot[ipt]])
                    energies_ref.append([E_ref['tot'][ipt]])
        rmsd_values = calculate_rmsd(energy, energy_ref)
        rmsd_midrange_value = calculate_rmsd(energies_pred, energies_ref)
        # print(key, '%.3f'%rmsd_values[0])
    return np.array(energy), np.array(energy_ref), np.array(rmsd_values), np.array(rmsd_midrange_value), np.array(shift)

In [5]:
# get params or restart from fitted params
def get_params(restart, params0):
    comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']
    if restart is None:
        params = {}
        sr_forces = {
                'ex': 'SlaterExForce',
                'es': 'SlaterSrEsForce',
                'pol': 'SlaterSrPolForce',
                'disp': 'SlaterSrDispForce',
                'dhf': 'SlaterDhfForce',
                }
        for k in params0['ADMPPmeForce']:
            params[k] = params0['ADMPPmeForce'][k]
        for k in params0['ADMPDispPmeForce']:
            params[k] = params0['ADMPDispPmeForce'][k]
        for c in comps:
            if c == 'tot':
                continue
            force = sr_forces[c]
            for k in params0[sr_forces[c]]:
                if k == 'A':
                    params['A_'+c] = params0[sr_forces[c]][k]
                else:
                    params[k] = params0[sr_forces[c]][k]
        # a random initialization of A
        for c in comps:
            if c == 'tot':
                continue
            params['A_'+c] = jnp.array(np.random.random(params['A_'+c].shape))
        # specify charges for es damping
        params['Q'] = params0['QqTtDampingForce']['Q']
    else:
        with open(restart, 'rb') as ifile:
            params = pickle.load(ifile)
    return params

def params_convert(params):
    params_ex = {}
    params_sr_es = {}
    params_sr_pol = {}
    params_sr_disp = {}
    params_dhf = {}
    params_dmp_es = {}  # electrostatic damping
    params_dmp_disp = {} # dispersion damping
    for k in ['B']:
        params_ex[k] = params[k]
        params_sr_es[k] = params[k]
        params_sr_pol[k] = params[k]
        params_sr_disp[k] = params[k]
        params_dhf[k] = params[k]
        params_dmp_es[k] = params[k]
        params_dmp_disp[k] = params[k]
    if 'C' in params:
        for k in ['C']:
            params_ex[k] = params[k]
    if 'D' in params:
        for k in ['D']:
            params_ex[k] = params[k]
    params_ex['A'] = params['A_ex']
    params_sr_es['A'] = params['A_es']
    params_sr_pol['A'] = params['A_pol']
    params_sr_disp['A'] = params['A_disp']
    params_dhf['A'] = params['A_dhf']
    # damping parameters
    params_dmp_es['Q'] = params['Q']
    params_dmp_disp['C6'] = params['C6']
    params_dmp_disp['C8'] = params['C8']
    params_dmp_disp['C10'] = params['C10']
    p = {}
    p['SlaterExForce'] = params_ex
    p['SlaterSrEsForce'] = params_sr_es
    p['SlaterSrPolForce'] = params_sr_pol
    p['SlaterSrDispForce'] = params_sr_disp
    p['SlaterDhfForce'] = params_dhf
    p['QqTtDampingForce'] = params_dmp_es
    p['SlaterDampingForce'] = params_dmp_disp
    return p


In [6]:
def get_all_homo_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a == b and b in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

def get_all_contain_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a in arr and b in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

def get_either_contain_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a in arr or b in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test


In [7]:
# data_file = 'data_train_final_wt_lr.pickle'
data_file = '../../data/data_dimer.pickle'

with open(data_file, 'rb') as ifile:
    data = pickle.load(ifile)

In [8]:
@jit
def calculate_weights(E_tot_full, thresh):
    kT = 2.494  # 300 K = 2.494 kJ/mol
    weights_pts = jnp.piecewise(E_tot_full, [E_tot_full<thresh, E_tot_full>=thresh], [lambda x: jnp.array(1.0), lambda x: jnp.exp(-(x-thresh)/kT)])
    return weights_pts

ions = ['Li', 'Na', 'PF6', 'BOB', 'FSI', 'TFSI', 'BF4', 'DFP', 'DFOB']
dimer_repulsive = get_all_homo_key(data, ions)

for pair in data.keys():
    print(pair)
    if pair in dimer_repulsive:
        for sid in data[pair].keys():
            data[pair][sid]['wts'] = jnp.ones(12)
    else:        
        for sid in data[pair].keys():
            scan_res = data[pair][sid]
            E_tot_full = scan_res['tot_full']
            thresh = 25
            data[pair][sid]['wts'] = calculate_weights(E_tot_full, thresh)

conf_001_DMC_DMC
conf_003_EC_EC
conf_018_DMC_EC
conf_045_Li_Li
conf_047_PF6_PF6
conf_051_Li_PF6
conf_060_Li_DMC
conf_062_Li_EC
conf_078_PF6_DMC
conf_080_PF6_EC


In [9]:
dimer_train = get_all_contain_key(data, ['Li', 'PF6','DMC','EC'])
dimer_train.sort()
print(len(dimer_train),dimer_train)


10 ['conf_001_DMC_DMC', 'conf_003_EC_EC', 'conf_018_DMC_EC', 'conf_045_Li_Li', 'conf_047_PF6_PF6', 'conf_051_Li_PF6', 'conf_060_Li_DMC', 'conf_062_Li_EC', 'conf_078_PF6_DMC', 'conf_080_PF6_EC']


In [10]:
# get params or restart from fitted params
def get_params(restart, params0):
    comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']
    if restart is None:
        params = {}
        sr_forces = {
                'ex': 'SlaterExForce',
                'es': 'SlaterSrEsForce',
                'pol': 'SlaterSrPolForce',
                'disp': 'SlaterSrDispForce',
                'dhf': 'SlaterDhfForce',
                }
        for k in params0['ADMPPmeForce']:
            params[k] = params0['ADMPPmeForce'][k]
        for k in params0['ADMPDispPmeForce']:
            params[k] = params0['ADMPDispPmeForce'][k]
        for c in comps:
            if c == 'tot':
                continue
            force = sr_forces[c]
            for k in params0[sr_forces[c]]:
                if k == 'A':
                    params['A_'+c] = params0[sr_forces[c]][k]
                else:
                    params[k] = params0[sr_forces[c]][k]
        # a random initialization of A
        for c in comps:
            if c == 'tot':
                continue
            params['A_'+c] = jnp.array(np.random.random(params['A_'+c].shape)) * 100
        # specify charges for es damping
        params['Q'] = params0['QqTtDampingForce']['Q']
    else:
        with open(restart, 'rb') as ifile:
            params = pickle.load(ifile)
    return params



In [11]:
restart = None
ff = 'phyneo_ecl.xml'
params0 = Hamiltonian(ff).getParameters()
params = get_params(restart, params0)

In [12]:
class_instances = {}
cal_energy = {}    
MSELoss_grad = {} 

dimer_train.sort()
for pair in dimer_train: 
    if pair not in MSELoss_grad:
        print(pair)
        conf, numb_conf, monomer_A, monomer_B = pair.split('_')
        dimer_file = f'dimer_{numb_conf}_{monomer_A}_{monomer_B}'
        dimer_file = f'../../data/dimer_bank/{dimer_file}.pdb'
        pdb_A_file = f'../../data/pdb_bank/{monomer_A}.pdb'
        pdb_B_file = f'../../data/pdb_bank/{monomer_B}.pdb'
        class_instances[pair] = BasePairs(ff, dimer_file, pdb_A_file, pdb_B_file)
for class_name, class_instance in class_instances.items():
    cal_energy[class_name] = jit(vmap(class_instance.cal_E, in_axes=(None, 0, 0), out_axes=(0, 0, 0, 0, 0, 0)))

for key in dimer_train:
    batch = list(data[key].keys())[1]
    if key not in MSELoss_grad:
        def MSELoss(params, data):
            '''
            The weighted mean squared error loss function
            Conducted for each scan
            '''
            # batch = padding(batch)
            scan_res = data
            comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']
            weights_comps = jnp.array([0.1, 0.1, 0.1, 0.1, 0.1, 1.0])
            weights_pts = scan_res['wts']
            npts = len(weights_pts)

            energies = {
                    'ex': jnp.zeros(npts),
                    'es': jnp.zeros(npts),
                    'pol': jnp.zeros(npts),
                    'disp': jnp.zeros(npts),
                    'dhf': jnp.zeros(npts),
                    'tot': jnp.zeros(npts)
                    }

            E_ex, E_es, E_pol, E_disp, E_dhf, E_tot = cal_energy[key](params, scan_res['posA'], scan_res['posB'])
            
            for ipt in range(npts):
                energies['ex'] = energies['ex'].at[ipt].set(E_ex[ipt])
                energies['es'] = energies['es'].at[ipt].set(E_es[ipt])
                energies['pol'] = energies['pol'].at[ipt].set(E_pol[ipt])
                energies['disp'] = energies['disp'].at[ipt].set(E_disp[ipt])
                energies['dhf'] = energies['dhf'].at[ipt].set(E_dhf[ipt])
                energies['tot'] = energies['tot'].at[ipt].set(E_tot[ipt])

            errs = jnp.zeros(len(comps))
            for ic, c in enumerate(comps):
                dE = scan_res[c] - energies[c] 
                mse = dE**2 * weights_pts / jnp.sum(weights_pts)
                errs = errs.at[ic].set(jnp.sum(mse))
            loss = jnp.sum(weights_comps * errs)
            return loss
    
    MSELoss_grad[key] = jit(value_and_grad(MSELoss, argnums=(0)))
    err, gradients = MSELoss_grad[key](params, data[key][batch])
    print(key , err)



conf_001_DMC_DMC
conf_003_EC_EC
conf_018_DMC_EC
conf_045_Li_Li
conf_047_PF6_PF6
conf_051_Li_PF6
conf_060_Li_DMC
conf_062_Li_EC
conf_078_PF6_DMC
conf_080_PF6_EC
conf_001_DMC_DMC 14383.838067658842
conf_003_EC_EC 5031.898362997387
conf_018_DMC_EC 2087.588885310546
conf_045_Li_Li 28.50754013180081
conf_047_PF6_PF6 1089913.156000765
conf_051_Li_PF6 588439.7990292178
conf_060_Li_DMC 16231.81822993313
conf_062_Li_EC 8566.228099954364
conf_078_PF6_DMC 75709.30476266344
conf_080_PF6_EC 149538.26624589565


In [13]:
trunk = []
for key in dimer_train:
    for batch in data[key]:
        trunk.append([key,batch])

os.makedirs('params', exist_ok=True)
save_model = 'params/params.pickle'

def mask_fn(grads):
    for k in grads:
        if k.startswith('A_') or k == 'B':
            continue
        else:
            grads[k] = 0.0
    return grads

lr = 0.1
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)
n_epochs = 1000

loss_train = []
loss_test = []
for i_epoch in range(n_epochs):
    np.random.shuffle(trunk)
    for key0, batch in trunk:
        loss, grads = MSELoss_grad[key0](params, data[key0][batch])
        grad = mask_fn(grads)
        updates, opt_state = optimizer.update(grad, opt_state)
        params = optax.apply_updates(params, updates)

    print(f"{i_epoch} {loss:.6f} {key0}")
        
    if (i_epoch % 10 == 0):
        with open(save_model, 'wb') as ofile:
            pickle.dump(params, ofile)

0 7025.228809 conf_078_PF6_DMC
1 691.969905 conf_060_Li_DMC


KeyboardInterrupt: 